# How ForEachBatch Solves the Problem of Writing to Multiple Sinks Simultaneously in Spark Structured Streaming
* Topic: How ForEachBatch Solves the Problem of Writing to Multiple Sinks Simultaneously in Spark Structured Streaming
* Author: Oindrila Chakraborty

# Write Streaming Data to a Kafka Topic as Sink
* To write <b>Streaming</b> data to a <b>Kafka Topic</b> using <b>Spark Structured Streaming</b> on Databricks, the outbound <b>DataFrame</b> must contain a <b>value</b> column, and, optionally a <b>key</b> column.
* Both of these columns should be cast to either <b>STRING</b>, or, <b>BINARY</b> format.


In [0]:
# STEP 1: Define a Hardcoded DDL-Formatted String Schema.
schema_ddl = "STRUCT<employee_id STRING, data STRUCT<assignments: ARRAY<STRUCT<assignment_id: STRING, start_from: STRING, end_at: STRING, status: STRING>>>, event_id STRING, event_offset BIGINT, event_publisher STRING, event_time STRING>"

# STEP 2: Connect to the Kafka Topic and Launch the Real-Time Streaming Engine
streaming_df = (spark.readStream
                    .format("kafka")
                    .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
                    .option("subscribe", "topic_name")
                    .load()
)

# STEP 3: Parse Message Values Seamlessly via the DDL-Formatted String Schema
parsed_stream_df = (streaming_df
                            .select(from_json(col("value").cast("string"), schema_ddl).alias("data"))
                            .select(col("key"), "data.*")
)

# STEP 4: Write to the Target Kafka Topic
target_query = (parsed_stream_df.writeStream
                                .format("kafka")
                                .option("kafka.bootstrap.servers", "<TARGET_BROKER_IP>:<PORT>")
                                .option("topic", "target_topic_name")
                                .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_locations/target_kafka_topic")
                                .start()
)

# Problem of Writing Multiple "Write Stream" Queries to Write Streamimg Data to Multiple Sinks Simultaneously in Spark Structured Streaming
* Consider a scenario, where messages are read from a <b>Kafka Topic</b> using <b>Read Stream Query</b> to a <b>PySpark DataFrame</b>, like -
    * <b>topic_names = "topic_a, topic_b"</b>
    <br>
    <br><b>df = (<b>
    <br><b>spark.readStream</b>
    <br><b>         .format("kafka")</b>
    <br><b>         .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")</b>
    <br><b>         .option("subscribe", topic_names)</b>
    <br><b>         .load()</b>
    <br><b>)</b>
* Now, to write the data from the same <b>DataFrame</b>, i.e., <b>df</b>, into multiple <b>Sinks</b>, say, to another <b>Kafka Topic</b>, and, a <b>PostgreSql Table</b>, two separate <b>Write Stream Queries</b> are used -
    * <b>Write Stream for Kafka Topic</b>:
    * <b>Write Stream for PostgreSql Table</b>: